# Julia notebook to compute the symbolic solution to the frictional geostrophic equations with a specified relaxation buoyancy field and surface wind stress.
twnh 

Dec '25   Write `ExampleSolution_v0.5.ipynb` <br>
Jul '26.  Write `FrictionalGeostrophicEquationSolution.ipynb`

This notebook solves the non-dimensional, linear, time-independent frictional geostrophic equations:

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + Ek \,  \frac{d}{dz} \left( \nu \frac{d u}{d z} \right), \\
  f  u & = - \frac{\partial p}{\partial y} + Ek \,  \frac{d}{dz} \left( \nu \frac{d v}{d z} \right) , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu = \nu (z)$ (customized so $\nu = 1$ at the moment), and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \nu \partial_z u & = \tau^x \\
\displaystyle  \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the applied surface stress.
The pressure $p$ is due to the surface pressure field $p_s(x,y)$ and buoyancy field $b(x,y,z)$:

\begin{align}
p(x,y,z) & = p_s(x,y) + \int_{z}^{0} b(x,y,z') \; dz' , \\
\implies p_b(x, y) & \equiv p(x, y, z=-H(x,y)) = p_s(x,y) + \int_{-H(x,y)}^{0} b(x,y,z') \; dz' .
\end{align}

This notebook also derives the equation satisfied by the surface pressure field $p_s(x,y)$.

In [1]:
using SymPy
# const I = SymPy.I ;   # Define I as the imaginary unit

### Define symbols:

In [2]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true)   # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                  # Horizontal coordinates
H      = SymFunction("H", real=true, positive=true)   # Domain depth H(x,y)
geometry_params = (H(x,y), z, ξ)

Ek     = symbols("Ek ",   real=true, positive=true)   # Ekman number

# Frictional geostrophic equations symbolic parameters:
f      = symbols("f",       real=true, positive=true) # Coriolis parameter
ϕ      = symbols("ϕ",       real=true, positive=true) # Viscosity parameter
τs     = SymFunction("τs",  complex=true)             # Complex surface stress
psg    = SymFunction("psg", complex=true)             # Surface pressure gradient (∂/∂x + i ∂/∂y) pₛ(x,y)

# Buoyancy symbolic parameters:
b      = SymFunction("b", real=true)                # Buoyancy field b(x,y,z)
ψ      = symbols("ψ",     real=true, positive=true) # Diffusivity parameters

# Define viscosity profile here:
ν = 1.0 + 0.0 * z ;                                   # Constant viscosity profile. Aspire to depth-varying profile in future.
uv_params = (f, Ek, ν) 

# Convenience variables:
root_iϕ = ϕ*sympy.sqrt(im) ;

### Compute pressure fields for the specified buoyancy field $b$:

In [3]:
# Define buoyancy gradient field:
bg     = SymFunction("bg",    complex=true)         # Buoyancy gradient

# Compute pressure field:
pbarog = SymPy.integrate(bg(x,y,ξ),(ξ,z,0))         # Baroclinic presure gradient 
pg     = psg(x,y) + pbarog                          # Total pressure gradient
pbotg  = pg.subs(z,-H(x,y))                         # Bottom pressure gradient

               0                   
               ⌠                   
psg(x, y) +    ⎮     bg(x, y, ξ) dξ
               ⌡                   
            -H(x, y)               

### Function to solve the frictional geostrophic equation using a Green's function: This is where `SymPy` is unavoidable as there's no `Symbolics` equivalent to `dsolve` as of July 2026.

In [4]:
function compute_Guv(uv_params, geometry_params)
    # Setup symbols and parameters:
    f, Ek, ν = uv_params
    H, z, ξ = geometry_params
    uv      = SymFunction("uv")
    A       = symbols("A",real=true)                                                  # Unknown coefficient in the Green's function solution

    # 0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(-im * f * uv(z) + Ek * diff(ν * diff(uv(z),z),z), 0)
    
    # 1. Solve for $G_- (z)$ on $-H \le z \le ξ$:
    Gₘ = dsolve(ode, uv(z), ics = Dict(uv(z).subs(z,-H(x,y))=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(uv(z),Gₘ)) == 0                               # Check solution
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1],A)
    @assert sympy.simplify(Gₘ.subs(z,-H(x,y))) == 0                                   # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $ξ \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics = Dict(diff(uv(z),z).subs(z,0)=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(uv(z),Gₚ)) == 0                               # Check solution
    @assert sympy.simplify(diff(Gₚ,z).subs(z,0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z)

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = Gₘ * Gₚ.subs(z,ξ) / (ν.subs(z,ξ) * W.subs(z,ξ))
    Gp = Gₘ.subs(z,ξ) * Gₚ / (ν.subs(z,ξ) * W.subs(z,ξ))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert sympy.simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ν.subs(z,ξ)) == 0

    # #5. Define piecewise Green's function:
    G = sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ)))
    
    # Check boundary conditions:
    @assert sympy.simplify(diff(G,z).subs(z,0).subs(ξ,-H//2)) == 0
    @assert sympy.simplify(G.subs(z,-H(x,y)).subs(ξ,-H//2)) == 0

    # Final sympy.simplify (to cancel constants). Avoid sympy.simplify in general because it's not always reproducible.
    G = sympy.simplify(G)
    return G
end ;

### Compute the velocity Green's function:

In [5]:
Guv_sym = compute_Guv(uv_params,geometry_params) 
Guv = Guv_sym.subs(f,ϕ^2 * Ek)
println("1.1 Symbolic computation of Guv completed.")
display(Guv)

1.1 Symbolic computation of Guv completed.


⎧     3/2 ⎛   2⋅√ⅈ⋅ξ⋅ϕ    2⋅√ⅈ⋅ϕ⋅(z + H(x, y))    2⋅√ⅈ⋅ϕ⋅(z + ξ + H(x, y))     ↪
⎪0.5⋅ⅈ   ⋅⎝- ℯ         + ℯ                     + ℯ                         - 1 ↪
⎪───────────────────────────────────────────────────────────────────────────── ↪
⎪                                     ⎛ 2⋅√ⅈ⋅ϕ⋅H(x, y)    ⎞                    ↪
⎪                                   ϕ⋅⎝ℯ               + 1⎠                    ↪
⎨                                                                              ↪
⎪     3/2 ⎛   2⋅√ⅈ⋅z⋅ϕ    2⋅√ⅈ⋅ϕ⋅(ξ + H(x, y))    2⋅√ⅈ⋅ϕ⋅(z + ξ + H(x, y))     ↪
⎪0.5⋅ⅈ   ⋅⎝- ℯ         + ℯ                     + ℯ                         - 1 ↪
⎪───────────────────────────────────────────────────────────────────────────── ↪
⎪                                     ⎛ 2⋅√ⅈ⋅ϕ⋅H(x, y)    ⎞                    ↪
⎩                                   ϕ⋅⎝ℯ               + 1⎠                    ↪

↪ ⎞  -√ⅈ⋅ϕ⋅(z + ξ)           
↪ ⎠⋅ℯ                        
↪ ────────────────  for z ≤ ξ
↪                 

In [6]:
# Used for stress-driven part of the velocity:
Guv0 = Guv.subs(ξ,0)
Latex_Guv0 = (im * sympy.sqrt(im) * exp(-root_iϕ * z)/(ϕ * (1 + exp(2 * root_iϕ * H(x,y))))) *
(exp(2 * root_iϕ * (z + H(x,y))) - 1)
resid = Guv0 - Latex_Guv0
@assert resid == 0
println("1.2 LaTeX Guv0 expression matches.")
display(Latex_Guv0)

# Used for surface-pressure-driven part of the velocity:
tmp = sympy.integrate(sympy.expand(Guv), (ξ, -H(x,y), 0)).args[1].args[1]
max_obj = sympy.Max(z, -H(x, y))
Guv_int_wrt_ξ = tmp.subs(max_obj, z)
Latex_Guv_int_wrt_ξ = (im / (ϕ^2 * (1 + exp(2 * root_iϕ * H(x,y)))) ) *
(1 + exp(2 * root_iϕ * H(x,y)) - exp(root_iϕ * (z + H(x,y))) - exp(root_iϕ * (H(x,y) - z)))
resid = Guv_int_wrt_ξ - Latex_Guv_int_wrt_ξ
@assert resid == 0
println("1.3 LaTeX Guv integral wrt ξ expression matches.")
display(Latex_Guv_int_wrt_ξ)

     ⎛ 2⋅√ⅈ⋅ϕ⋅(z + H(x, y))    ⎞  -√ⅈ⋅z⋅ϕ
√ⅈ⋅ⅈ⋅⎝ℯ                     - 1⎠⋅ℯ       
─────────────────────────────────────────
           ⎛ 2⋅√ⅈ⋅ϕ⋅H(x, y)    ⎞         
         ϕ⋅⎝ℯ               + 1⎠         

1.2 LaTeX Guv0 expression matches.
1.3 LaTeX Guv integral wrt ξ expression matches.


  ⎛   √ⅈ⋅ϕ⋅(-z + H(x, y))    √ⅈ⋅ϕ⋅(z + H(x, y))    2⋅√ⅈ⋅ϕ⋅H(x, y)    ⎞
ⅈ⋅⎝- ℯ                    - ℯ                   + ℯ               + 1⎠
──────────────────────────────────────────────────────────────────────
                        2 ⎛ 2⋅√ⅈ⋅ϕ⋅H(x, y)    ⎞                       
                       ϕ ⋅⎝ℯ               + 1⎠                       

In [7]:
# Used for bottom boundary stress computation:
tmp  = diff(Guv, z)
tmp2 = tmp.subs(z, -H(x,y))
dGuv_dz_at_bottom = tmp2.args[1].args[1]
Latex_dGuv_dz_at_bottom = (-exp(-root_iϕ * (ξ - H(x,y)))) / (1 + exp(2 * root_iϕ * H(x,y))) * (1 + exp(2 * root_iϕ * ξ))
resid = dGuv_dz_at_bottom - Latex_dGuv_dz_at_bottom
@assert resid == 0
println("1.4 LaTeX d/dz Guv(z,ξ) @ z = -H(x,y) matches.")
display(Latex_dGuv_dz_at_bottom)

Latex_dGuv_dz_at_bottom_ξ0 = Latex_dGuv_dz_at_bottom.subs(ξ,0)
resid = dGuv_dz_at_bottom.subs(ξ,0) - Latex_dGuv_dz_at_bottom_ξ0
@assert resid == 0
println("1.5 LaTeX d/dz Guv(z,0) @ z = -H(x,y) matches.")
display(Latex_dGuv_dz_at_bottom_ξ0)

 ⎛ 2⋅√ⅈ⋅ξ⋅ϕ    ⎞  -√ⅈ⋅ϕ⋅(ξ - H(x, y)) 
-⎝ℯ         + 1⎠⋅ℯ                    
──────────────────────────────────────
          2⋅√ⅈ⋅ϕ⋅H(x, y)              
         ℯ               + 1          

     √ⅈ⋅ϕ⋅H(x, y)  
 -2⋅ℯ              
───────────────────
 2⋅√ⅈ⋅ϕ⋅H(x, y)    
ℯ               + 1

1.4 LaTeX d/dz Guv(z,ξ) @ z = -H(x,y) matches.
1.5 LaTeX d/dz Guv(z,0) @ z = -H(x,y) matches.


In [8]:
# Used for baroclinic-pressure-driven part of the velocity:
tmp = sympy.integrate(sympy.expand(Guv * exp(ψ * ξ)),(ξ,-H(x,y),0))
Guv_exp_psi_xi_int_wrt_ξ = tmp.subs(max_obj, z)
Latex_Guv_exp_psi_xi_int_wrt_ξ = (1/(ψ^2 - im * ϕ^2)) * (
    ( im * sympy.sqrt(im) * ψ * exp(      ψ * H(x,y)) * (exp(2 * root_iϕ * (z + H(x,y))) - 1) -
                            ϕ * exp(root_iϕ * H(x,y)) * (exp(2 * root_iϕ *  z          ) + 1)
    ) * exp(-root_iϕ * z - ψ * H(x,y))
    /(ϕ * (1 + exp(2 * root_iϕ * H(x,y))))
    + exp(ψ * z)
)
resid = sympy.expand(Guv_exp_psi_xi_int_wrt_ξ - Latex_Guv_exp_psi_xi_int_wrt_ξ)
@assert resid == 0
println("1.6 LaTeX Guv * exp(ψ * z) integral matches.")
display(Latex_Guv_exp_psi_xi_int_wrt_ξ)

1.6 LaTeX Guv * exp(ψ * z) integral matches.


       ⎛       ⎛ 2⋅√ⅈ⋅ϕ⋅(z + H(x, y))    ⎞  ψ⋅H(x, y)     ⎛ 2⋅√ⅈ⋅z⋅ϕ    ⎞  √ⅈ⋅ ↪
 z⋅ψ   ⎝√ⅈ⋅ⅈ⋅ψ⋅⎝ℯ                     - 1⎠⋅ℯ          - ϕ⋅⎝ℯ         + 1⎠⋅ℯ    ↪
ℯ    + ─────────────────────────────────────────────────────────────────────── ↪
                                                ⎛ 2⋅√ⅈ⋅ϕ⋅H(x, y)    ⎞          ↪
                                              ϕ⋅⎝ℯ               + 1⎠          ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                   2      2                    ↪
                                                  ψ  - ⅈ⋅ϕ                     ↪

↪ ϕ⋅H(x, y)⎞  -√ⅈ⋅z⋅ϕ - ψ⋅H(x, y)
↪          ⎠⋅ℯ                   
↪ ───────────────────────────────
↪                                
↪                                
↪ ───────────────────────────────
↪                                
↪                                

### Symbolic computation of flow $(u(z),v(z)), U, V, \tau_b$

In [ ]:
# Compute flow field u(z), v(z):
𝔲1 = SymPy.integrate(sympy.expand(Guv * pg.subs(z,ξ)),(ξ,-H(x,y),0))
𝔲1 = 𝔲1.subs(max_obj,z)             # Buoyancy-driven part of the velocity
𝔲2 = - Guv0 * τs(x,y)               # Stress-driven part of the velocity
𝔲 = 𝔲1 + 𝔲2
println("2.1 Symbolic computation of 𝔲 completed.")
display(𝔲)

2.1 Symbolic computation of 𝔲 completed.


                                                                 ⎛             ↪
                                                                 ⎜0            ↪
                                                                 ⎜⌠            ↪
     3/2 ⎛ √ⅈ⋅z⋅ϕ  √ⅈ⋅ϕ⋅H(x, y)    ⎞ ⎛ √ⅈ⋅z⋅ϕ  √ⅈ⋅ϕ⋅H(x, y)    ⎞ ⎜⎮            ↪
0.5⋅ⅈ   ⋅⎝ℯ      ⋅ℯ             - 1⎠⋅⎝ℯ      ⋅ℯ             + 1⎠⋅⎜⎮ psg(x, y)⋅ ↪
                                                                 ⎜⌡            ↪
                                                                 ⎜             ↪
                                                                 ⎜             ↪
                                                                 ⎝             ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                               ↪
                                                                               ↪

↪                          

In [ ]:

# Check that the final expression for 𝔲 satisfies the original differential equation:
tmp1 = -im * f * 𝔲1 + Ek * diff(ν * diff(𝔲1,z),z)
tmp1 = sympy.simplify(tmp1.subs(ϕ,sqrt(f/Ek)))
tmp2 = -im * f * 𝔲2 + Ek * diff(ν * diff(𝔲2,z),z)
tmp2 = sympy.simplify(tmp2.subs(ϕ,sqrt(f/Ek)))
@assert tmp1 + tmp2 == pg
println("2.2 Final expression for 𝔲 satisfies the original differential equation.")

# Compute depth-integrated flow:
𝔘 = SymPy.integrate(sympy.expand(𝔲),(z,-H(x,y),0))

# Compute bottom stress on fluid:
τb = - ν * diff(𝔲,z).subs(z,-H(x,y))
    
# Check expression for surface stress on fluid:
@assert sympy.simplify(diff(𝔲1,z).subs(z,0)) == 0     # Pressure-driven part of surface stress vanishes
tmp = ν * diff(𝔲,z).subs(z,0)
@assert sympy.simplify(tmp - τs(x,y)) == 0
println("2.3 Expression for surface stress on fluid is correct.")
display(τb)

AssertionError: AssertionError: tmp1 + tmp2 == pg

In [14]:
tmp2

0